# 01 - Client Market Report

Notebook nay doc cac output da duoc pipeline sinh san. No khong chay lai pipeline va khong ghi de len output production.

Recommended flow truoc khi mo notebook:

```bash
python scripts/run_daily_pipeline.py --online --update-index --build-portfolio
```

Neu SSI rate limit:

```bash
python scripts/run_daily_pipeline.py --offline --build-portfolio
```

In [ ]:
from pathlib import Path
import json
from datetime import datetime

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data").exists() and (candidate / "src").exists() and (candidate / "scripts").exists():
            return candidate
    return Path.cwd()


ROOT = find_project_root()
OUTPUT_DIR = ROOT / "data" / "output"
PROCESSED_DIR = ROOT / "data" / "processed"
missing_files = []


def load_parquet_safe(relative_path: str) -> pd.DataFrame:
    path = ROOT / relative_path
    if not path.exists():
        missing_files.append(relative_path)
        return pd.DataFrame()
    try:
        return pd.read_parquet(path)
    except Exception as exc:
        missing_files.append(f"{relative_path} (read_error: {exc})")
        return pd.DataFrame()


def load_json_safe(relative_path: str) -> dict:
    path = ROOT / relative_path
    if not path.exists():
        missing_files.append(relative_path)
        return {}
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception as exc:
        missing_files.append(f"{relative_path} (read_error: {exc})")
        return {}


def latest_date(df: pd.DataFrame, columns=("date", "analysis_as_of", "source_latest_date")):
    if df.empty:
        return None
    for column in columns:
        if column not in df.columns:
            continue
        values = pd.to_datetime(df[column], errors="coerce").dropna()
        if not values.empty:
            return values.max().date().isoformat()
    return None


def choose_weight_column(df: pd.DataFrame) -> str | None:
    for column in ["risk_parity", "hrp", "inverse_volatility", "equal_weight"]:
        if column in df.columns:
            return column
    return None


def pct_text(value):
    if value is None or pd.isna(value):
        return "N/A"
    return f"{value * 100:.2f}%"


client_ready = load_parquet_safe("data/output/client_ready_watchlist.parquet")
trading_watchlist = load_parquet_safe("data/output/trading_watchlist.parquet")
research_candidates = load_parquet_safe("data/output/research_candidates.parquet")
portfolio_weights = load_parquet_safe("data/output/portfolio_weights.parquet")
capital_plan = load_parquet_safe("data/output/capital_plan.parquet")
stock_features = load_parquet_safe("data/processed/stock_features.parquet")
benchmark_vnindex = load_parquet_safe("data/processed/benchmark_vnindex.parquet")
pipeline_metadata = load_json_safe("data/output/pipeline_metadata.json")
portfolio_stack_metadata = load_json_safe("data/output/portfolio_stack_metadata.json")
portfolio_risk_report = load_json_safe("data/output/portfolio_risk_report.json")

report_generated_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
display(Markdown(f"Project root detected: `{ROOT}`"))
if missing_files:
    display(Markdown("**Missing / unreadable inputs:** " + ", ".join(f"`{item}`" for item in missing_files)))
else:
    display(Markdown("**All primary report inputs were loaded successfully.**"))

## 1. Title & Run Information

In [ ]:
run_info = pd.DataFrame(
    [
        ("Report generated at", report_generated_at),
        ("Pipeline run timestamp", pipeline_metadata.get("run_timestamp", "N/A")),
        ("Pipeline mode", pipeline_metadata.get("run_mode", "N/A")),
        ("Freshness status", pipeline_metadata.get("freshness_status", "N/A")),
        ("Latest stock date", pipeline_metadata.get("latest_stock_date", latest_date(stock_features) or "N/A")),
        ("Latest benchmark date", pipeline_metadata.get("latest_benchmark_date", latest_date(benchmark_vnindex) or "N/A")),
        ("Portfolio stack status", pipeline_metadata.get("portfolio_stack", {}).get("status", "DISABLED")),
        ("Online update status", pipeline_metadata.get("online_update", {}).get("status", "SKIPPED")),
    ],
    columns=["Metric", "Value"],
)
run_info

## 2. Executive Summary

In [ ]:
watchlist_counts = pipeline_metadata.get("watchlist_counts", {})
research_count = watchlist_counts.get("research_candidates", {}).get("rows", len(research_candidates))
trading_count = watchlist_counts.get("trading_watchlist", {}).get("rows", len(trading_watchlist))
client_ready_count = watchlist_counts.get("client_ready_watchlist", {}).get("rows", len(client_ready))
portfolio_enabled = pipeline_metadata.get("portfolio_stack", {}).get("enabled", False)
portfolio_status = pipeline_metadata.get("portfolio_stack", {}).get("status", "DISABLED")
warnings = list(pipeline_metadata.get("warnings", []))
online_update = pipeline_metadata.get("online_update", {})

summary_lines = [
    f"- Freshness status: **{pipeline_metadata.get('freshness_status', 'N/A')}**.",
    f"- Research universe currently contains **{research_count}** candidates.",
    f"- Trading watchlist currently contains **{trading_count}** names.",
    f"- Client-ready watchlist currently contains **{client_ready_count}** names.",
    f"- Portfolio stack status: **{portfolio_status}** (enabled={portfolio_enabled}).",
]
if online_update.get("status") not in {None, "OK", "SKIPPED"}:
    summary_lines.append(
        f"- Online update warning: **{online_update.get('error_category', 'UNKNOWN_ERROR')}** - {online_update.get('error_message', 'No detail')}"
    )
if warnings:
    summary_lines.append("- Key warnings: " + "; ".join(str(item) for item in warnings[:5]))
if missing_files:
    summary_lines.append("- Missing inputs detected: " + ", ".join(str(item) for item in missing_files))

display(Markdown("\n".join(summary_lines)))

## 3. Market Context

In [ ]:
if benchmark_vnindex.empty:
    display(Markdown("Benchmark VNINDEX data is missing, so market context is limited."))
else:
    bench = benchmark_vnindex.copy()
    bench["date"] = pd.to_datetime(bench["date"], errors="coerce")
    bench = bench.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)
    if "return_1d" not in bench.columns and "close" in bench.columns:
        bench["return_1d"] = bench["close"].pct_change(1)
    for period in [5, 10, 20]:
        col = f"return_{period}d"
        if col not in bench.columns and "close" in bench.columns:
            bench[col] = bench["close"].pct_change(period)
    latest = bench.iloc[-1]
    rolling_peak = bench["close"].cummax() if "close" in bench.columns else pd.Series(dtype=float)
    latest_drawdown = None
    if not rolling_peak.empty:
        latest_drawdown = float((bench["close"] / rolling_peak - 1.0).iloc[-1])
    context_rows = [
        ("Latest close", latest.get("close", "N/A")),
        ("5D return", pct_text(latest.get("return_5d"))),
        ("10D return", pct_text(latest.get("return_10d"))),
        ("20D return", pct_text(latest.get("return_20d"))),
        ("Latest drawdown vs rolling peak", pct_text(latest_drawdown)),
    ]
    display(pd.DataFrame(context_rows, columns=["Metric", "Value"]))
    fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
    axes[0].plot(bench["date"], bench["close"], color="navy", linewidth=1.8)
    axes[0].set_title("VNINDEX Close")
    axes[0].grid(alpha=0.25)
    axes[1].plot(bench["date"], bench.get("return_1d", pd.Series(index=bench.index, dtype=float)).fillna(0.0), color="darkred", linewidth=1.0)
    axes[1].axhline(0.0, color="black", linewidth=0.8, linestyle="--")
    axes[1].set_title("VNINDEX Daily Return")
    axes[1].grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

## 4. Market Breadth / Regime

In [ ]:
if stock_features.empty:
    display(Markdown("Stock feature data is missing, so breadth and regime review cannot be computed."))
else:
    stock = stock_features.copy()
    stock["date"] = pd.to_datetime(stock["date"], errors="coerce")
    stock = stock.dropna(subset=["date"]).sort_values(["date", "symbol"])
    latest_stock_date = stock["date"].max()
    latest_slice = stock[stock["date"] == latest_stock_date].copy()
    breadth_rows = []
    if "return_1d" in latest_slice.columns:
        breadth_rows.extend([
            ("Advancers", int((latest_slice["return_1d"] > 0).sum())),
            ("Decliners", int((latest_slice["return_1d"] < 0).sum())),
            ("Unchanged / NA", int((latest_slice["return_1d"].fillna(0) == 0).sum())),
        ])
    else:
        breadth_rows.append(("Advancers / Decliners", "Column return_1d is not available"))
    if "above_ma20" in latest_slice.columns:
        breadth_rows.append(("Pct above MA20", pct_text(latest_slice["above_ma20"].mean())))
    else:
        breadth_rows.append(("Pct above MA20", "Column above_ma20 is not available"))
    if "above_ma50" in latest_slice.columns:
        breadth_rows.append(("Pct above MA50", pct_text(latest_slice["above_ma50"].mean())))
    else:
        breadth_rows.append(("Pct above MA50", "Column above_ma50 is not available"))
    regime_note = "No regime column found in stock_features."
    for regime_col in ["market_regime", "volatility_regime", "regime"]:
        if regime_col in latest_slice.columns:
            counts = latest_slice[regime_col].fillna("Unknown").value_counts().to_dict()
            regime_note = ", ".join(f"{k}: {v}" for k, v in counts.items())
            break
    display(pd.DataFrame(breadth_rows, columns=["Metric", "Value"]))
    display(Markdown(f"**Regime note:** {regime_note}"))

## 5. Watchlist Overview and 6. Client-Ready Watchlist

In [ ]:
overview = pd.DataFrame(
    [
        ("Research candidates", len(research_candidates), latest_date(research_candidates) or "N/A"),
        ("Trading watchlist", len(trading_watchlist), latest_date(trading_watchlist) or "N/A"),
        ("Client-ready watchlist", len(client_ready), latest_date(client_ready) or "N/A"),
    ],
    columns=["Bucket", "Count", "Latest Date"],
)
overview

if client_ready.empty:
    display(Markdown("Client-ready watchlist is empty. Review pipeline warnings and candidate filters before sending to the client."))
else:
    important_cols = [
        "rank", "symbol", "date", "close", "final_score", "candidate_tier", "confidence_label",
        "volatility_regime", "risk_reward_estimate", "reason", "exclusion_reason"
    ]
    show_cols = [col for col in important_cols if col in client_ready.columns]
    display(client_ready[show_cols].head(10))
    bullet_lines = []
    for _, row in client_ready.head(10).iterrows():
        reason = row.get("reason", "No qualitative reason field available")
        tier = row.get("candidate_tier", "N/A")
        confidence = row.get("confidence_label", "N/A")
        regime = row.get("volatility_regime", "N/A")
        exclusion = row.get("exclusion_reason", "")
        warning_note = f" Caution: {exclusion}." if isinstance(exclusion, str) and exclusion.strip() else ""
        bullet_lines.append(
            f"- **{row.get('symbol', 'N/A')}** | tier: **{tier}** | confidence: **{confidence}** | regime: **{regime}**. Main strength: {reason}.{warning_note}"
        )
    display(Markdown("\n".join(bullet_lines)))

## 7. Portfolio Construction and 8. Risk Management

In [ ]:
weight_col = choose_weight_column(portfolio_weights)
if portfolio_weights.empty or weight_col is None:
    display(Markdown("Portfolio weight file is missing or does not contain a recognized allocation column."))
else:
    portfolio_view = portfolio_weights[[col for col in ["symbol", weight_col, "source_latest_date"] if col in portfolio_weights.columns]].copy()
    portfolio_view = portfolio_view.sort_values(weight_col, ascending=False).reset_index(drop=True)
    display(Markdown(f"Using **{weight_col}** as the primary displayed allocation column. These weights should be interpreted as a risk-framework allocation, not an all-in instruction."))
    display(portfolio_view)
    plt.figure(figsize=(11, 4))
    plt.bar(portfolio_view["symbol"], portfolio_view[weight_col], color="slateblue")
    plt.title(f"Portfolio Weights - {weight_col}")
    plt.xticks(rotation=45)
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()
    if "sector" in portfolio_weights.columns:
        sector_view = portfolio_weights.groupby("sector", dropna=False)[weight_col].sum().reset_index().sort_values(weight_col, ascending=False)
        display(sector_view)
    else:
        display(Markdown("Sector grouping is not available in `portfolio_weights.parquet`."))

risk_keys = [
    "annual_return", "annual_volatility", "max_drawdown", "sharpe_ratio",
    "sortino_ratio", "beta", "var_95", "cvar_95", "effective_n", "allocator_used"
]
risk_rows = [(key, portfolio_risk_report.get(key, "N/A")) for key in risk_keys]
display(pd.DataFrame(risk_rows, columns=["Risk Metric", "Value"]))

if capital_plan.empty:
    display(Markdown("Capital plan is not available, so capital allocation review is limited."))
else:
    capital_cols = [
        "symbol", "trade_edge_score", "conviction_score_resolved", "volatility_regime",
        "risk_pct_effective", "quantity", "position_value", "position_pct_nav", "stop_price"
    ]
    capital_show = [col for col in capital_cols if col in capital_plan.columns]
    display(capital_plan[capital_show].sort_values(capital_show[0]).reset_index(drop=True))
    display(Markdown("Capital allocation should be interpreted as staged, risk-budgeted sizing rather than an all-in deployment plan."))

## 9. Scenario Analysis and 10. Client Action Plan

In [ ]:
bullish_text = (
    "- **Bullish scenario:** if VNINDEX confirms trend continuation, priority should shift toward the highest-conviction client-ready names,"
    " with scaling based on portfolio weights and capital-plan risk limits rather than aggressive full deployment."
)
base_text = (
    "- **Base scenario:** if the market stays range-bound or continues to accumulate, keep the research and trading watchlists active,"
    " but deploy only partial size into names with stable confidence labels and acceptable volatility regime."
)
bearish_text = (
    "- **Bearish scenario:** if support breaks or volatility expands, reduce gross exposure, tighten stop discipline,"
    " and move weaker watchlist names back into observation mode instead of forcing entries."
)

client_action_lines = [
    "- **Follow closely:** names already in the client-ready bucket with strong confidence and no major exclusion reason.",
    "- **Scale in gradually:** names supported by the portfolio weight framework and capital plan, especially where risk budgets remain modest.",
    "- **Wait for confirmation:** names still attractive on research merit but lacking a cleaner tactical setup or carrying a warning note.",
    "- **Reduce exposure when needed:** if benchmark momentum weakens materially, breadth narrows, or volatility regime deteriorates."
]

display(Markdown("\n".join([bullish_text, base_text, bearish_text, "", *client_action_lines])))

## 11. Disclaimer

- Bao cao nay chi nham muc dich tham khao va ho tro ra quyet dinh.
- Day khong phai la cam ket loi nhuan hay khuyen nghi mua ban chac chan.
- Nha dau tu can tu danh gia khau vi rui ro, muc tieu va dieu kien thanh khoan cua minh.
- Du lieu va xep hang co the thay doi theo dien bien thi truong va theo lan cap nhat pipeline moi nhat.